Covering tools specs, async orchestration, vector retrieval, cross encoders, and semantic caching.

**Dependencies**

In [1]:
!pip install groq pydantic sentence-transformers rank_bm25 numpy asyncio cachetools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.1 MB/s eta 0:00:00


In [19]:
import os
from google.colab import userdata
api_key = userdata.get('API_KEY')

### 1. Tool schema design & Tool call first pattern

Force model to issue a tool call even when the query looks conversation or off-topic, using `tool_choice = "required"`

In [20]:
import json
from groq import Groq

In [22]:
groq_client = Groq(api_key=api_key)

Define JSON Schemas for 3 tools

In [24]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Fetch current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City and state/country"}
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "db_lookup",
            "description": "Look up user info or records in database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Database query term or user ID"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate basic mathematical expressions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '12 * 45'"}
                },
                "required": ["expression"]
            }
        }
    }
]

In [25]:
query = "Hello! What is 42 multiplied by 18?"

Enforce 'Tool Call First' by setting tool_choice='required'

In [26]:
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": query}],
    tools=tools,
    tool_choice="required", # Forces tool execution instead of natural text response
    temperature=0.0
)

In [27]:
tool_call = response.choices[0].message.tool_calls[0]
print(f"Forced Tool Name: {tool_call.function.name}")
print(f"Parsed Arguments: {tool_call.function.arguments}")

Forced Tool Name: calculator
Parsed Arguments: {"expression":"42 * 18"}


### 2. Parallel tool calls & async execution

Groq’s high inference speed generates multiple tool calls in a single completion pass. You can parse these tool calls and execute them concurrently via `asyncio`.

In [28]:
import asyncio

Mock async backend functions

In [29]:
async def fetch_weather_api(city: str) -> str:
    await asyncio.sleep(0.5) # Simulate API latency
    mock_data = {"Tokyo": "18°C, Rain", "New York": "22°C, Sunny", "London": "14°C, Cloudy"}
    return mock_data.get(city, "20°C, Clear")

In [33]:
async def run_parallel_tools():
    prompt = "Compare the current weather in Tokyo, New York, and London."

    # Send request with weather tool definition
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        tools=[tools[0]], # Weather tool
        tool_choice="auto",
        temperature=0.0
    )

    tool_calls = response.choices[0].message.tool_calls
    print(f"Generated {len(tool_calls)} tool calls concurrently.\n")

    # Map tool calls to async tasks
    async def process_call(tc):
        args = json.loads(tc.function.arguments)
        location = args.get("location")
        result = await fetch_weather_api(location)
        print(f"[EXECUTED] {location} -> {result}")
        return {"tool_call_id": tc.id, "result": result}

    # Execute all tool calls asynchronously
    results = await asyncio.gather(*(process_call(tc) for tc in tool_calls))
    return results

Run event loop

In [34]:
await run_parallel_tools()

Generated 3 tool calls concurrently.

[EXECUTED] Tokyo -> 18°C, Rain
[EXECUTED] New York -> 22°C, Sunny
[EXECUTED] London -> 14°C, Cloudy


[{'tool_call_id': 'z1se9yvz9', 'result': '18°C, Rain'},
 {'tool_call_id': 'hcxatakrc', 'result': '22°C, Sunny'},
 {'tool_call_id': 'jjf2sbk1z', 'result': '14°C, Cloudy'}]

### 3. Retrieval: Hybrid Search & Cross-Encoder Re-ranking

Combine keyword matching (BM25) with dense vector search (SentenceTransformers), normalize scores, and apply a Cross-Encoder for strict top-3 ranking.

In [35]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

Document store and Encoders

In [36]:
documents = [
    "Groq LPUs are designed specifically for fast inference of Large Language Models.",
    "Python is a popular programming language widely used in AI and data science.",
    "RAG systems combine vector retrieval with generative models to ground responses.",
    "BM25 is a sparse lexical retrieval algorithm based on TF-IDF scoring.",
    "Dense retrievers use bi-encoder neural networks to project text into embedding space.",
    "Cross-encoders perform full self-attention over query-document pairs for accurate scoring."
]

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Tokenize for BM25

In [37]:
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

Compute Dense Embeddings

In [38]:
doc_embeddings = bi_encoder.encode(documents, normalize_embeddings=True)

In [41]:
def hybrid_rerank_search(query: str, top_k_rerank: int = 3):
    # --- Step A: BM25 Lexical Score ---
    tokenized_query = query.lower().split()
    bm25_scores = np.array(bm25.get_scores(tokenized_query))
    bm25_norm = (bm25_scores - bm25_scores.min()) / (np.ptp(bm25_scores) + 1e-6)

    # --- Step B: Cosine Dense Score ---
    query_emb = bi_encoder.encode(query, normalize_embeddings=True)
    dense_scores = np.dot(doc_embeddings, query_emb)
    dense_norm = (dense_scores - dense_scores.min()) / (np.ptp(dense_scores) + 1e-6)

    # --- Step C: Weighted Hybrid Score ---
    hybrid_scores = 0.3 * bm25_norm + 0.7 * dense_norm
    top_10_indices = np.argsort(hybrid_scores)[::-1][:10]
    top_10_docs = [documents[i] for i in top_10_indices]

    # --- Step D: Cross-Encoder Re-ranking ---
    pairs = [[query, doc] for doc in top_10_docs]
    ce_scores = cross_encoder.predict(pairs)

    # Sort top candidates by Cross-Encoder score
    reranked_indices = np.argsort(ce_scores)[::-1][:top_k_rerank]
    final_docs = [top_10_docs[i] for i in reranked_indices]

    return final_docs

Test

In [42]:
query = "How do high-speed LLM hardware units like LPUs work?"
results = hybrid_rerank_search(query)
print("Top 3 Re-ranked Docs:")
for idx, doc in enumerate(results, 1):
    print(f"{idx}. {doc}")

Top 3 Re-ranked Docs:
1. Groq LPUs are designed specifically for fast inference of Large Language Models.
2. Dense retrievers use bi-encoder neural networks to project text into embedding space.
3. RAG systems combine vector retrieval with generative models to ground responses.


### 4. Query Rewritting

Transform messy conversational inputs into precise, search-optimized queries before feeding them to the hybrid retrieval pipeline.

In [43]:
def rewrite_query(raw_query: str) -> str:
    prompt = f"""You are a search query optimizer for a technical database.
Rewrite the following user statement into a clear, concise keyword-dense search query.
Do NOT reply with conversational filler. Output ONLY the query string.

Raw input: "{raw_query}"
Optimized Search Query:"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.choices[0].message.content.strip()

In [44]:
messy_user_input = "uhhh that thing about hardware acceleration chip made for fast inference..."
clean_query = rewrite_query(messy_user_input)

In [45]:
print(f"Original: {messy_user_input}")
print(f"Rewritten: {clean_query}")

Original: uhhh that thing about hardware acceleration chip made for fast inference...
Rewritten: "hardware acceleration chip inference optimization"


Feed clean query directly to retrieval pipeline

In [46]:
rag_context = hybrid_rerank_search(clean_query)

### 5. Semantic Caching & Cost Analysis

Cache prompt responses using vector similarity. If a incoming query's embedding is $\ge 0.90$ similar to a cached query, bypass Groq completely.

In [47]:
from sentence_transformers import SentenceTransformer

In [48]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:619: RuntimeWarning: coroutine 'run_parallel_tools' was never awaited
  elif hasattr(self, name) and name not in self._parameters:


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [49]:
class SemanticCache:
    def __init__(self, threshold=0.90):
        self.threshold = threshold
        self.cache = [] # List of dicts: {"query": str, "embedding": np.ndarray, "response": str}
        self.calls_saved = 0
        self.total_queries = 0

    def get_or_set(self, query: str, fetch_llm_fn):
        self.total_queries += 1
        query_emb = encoder.encode(query, normalize_embeddings=True)

        # Check existing cache entries
        for entry in self.cache:
            similarity = np.dot(query_emb, entry["embedding"])
            if similarity >= self.threshold:
                self.calls_saved += 1
                print(f"[CACHE HIT] Similarity: {similarity:.4f} | Bypassing LLM Call.")
                return entry["response"]

        # Cache Miss -> Call Groq
        print("[CACHE MISS] Calling Groq API...")
        response = fetch_llm_fn(query)
        self.cache.append({
            "query": query,
            "embedding": query_emb,
            "response": response
        })
        return response

    def get_savings_metrics(self, cost_per_1k_tokens=0.0005):
        # Estimated metrics computation
        saved_ratio = (self.calls_saved / self.total_queries) * 100 if self.total_queries > 0 else 0
        print(f"\n--- Cache Metrics ---")
        print(f"Total Requests: {self.total_queries}")
        print(f"Cache Hits: {self.calls_saved} ({saved_ratio:.1f}% saved)")

In [50]:
cache = SemanticCache(threshold=0.90)

In [51]:
def call_groq_llm(q: str):
    return groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": q}],
        temperature=0.0
    ).choices[0].message.content

In [53]:
# Query 1 (Miss)
cache.get_or_set("What is a GPU?", call_groq_llm)

# Query 2 (Hit - Semantic equivalent)
cache.get_or_set("what GPUs are?", call_groq_llm)

cache.get_savings_metrics()

[CACHE HIT] Similarity: 1.0000 | Bypassing LLM Call.
[CACHE HIT] Similarity: 0.9516 | Bypassing LLM Call.

--- Cache Metrics ---
Total Requests: 4
Cache Hits: 2 (50.0% saved)


# Capstone: Scale RAG Simulator & Hallucination Detector

Simulate scale with chunking window management and evaluate grounding. If context lacks sufficient evidence, force the model to issue an explicit refusal.

In [54]:
def chunk_document(text: str, chunk_size: int = 150, overlap: int = 30) -> list[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
    return chunks

def grounded_rag_agent(user_claim: str, retrieved_context: list[str]) -> str:
    context_block = "\n".join([f"- {c}" for c in retrieved_context])

    system_prompt = """
You are an uncompromising Grounded RAG Inspector.
Your objective is to answer the user's prompt strictly based on the provided retrieved context.

STRICT RULES:
1. If the retrieved context DOES NOT contain enough evidence to answer the user query, output strictly: "I don't know."
2. Do NOT use outside knowledge. Do NOT hallucinate facts not present in context.
"""

    user_prompt = f"""
<context>
{context_block}
</context>

<query>
{user_claim}
</query>
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0
    )

    return response.choices[0].message.content.strip()

In [55]:
sample_docs = [
    "Project Apollo was designed to land humans on the Moon and bring them safely back to Earth.",
    "Apollo 11 launched on July 16, 1969, carrying Neil Armstrong, Buzz Aldrin, and Michael Collins."
]

print("Test 1: Supported Fact")
ans1 = grounded_rag_agent("When did Apollo 11 launch?", sample_docs)
print(f"Answer: {ans1}\n")

print("Test 2: Out of Context Fact (Hallucination Test)")
ans2 = grounded_rag_agent("What was the fuel type of Apollo 11 Saturn V rocket?", sample_docs)
print(f"Answer: {ans2}")

Test 1: Supported Fact
Answer: Apollo 11 launched on July 16, 1969.

Test 2: Out of Context Fact (Hallucination Test)
Answer: I don't know.
